# Verona E-Commerce — Data Cleaning Walkthrough

This notebook demonstrates the data cleaning process interactively (the production cleaning logic lives in `src/data_cleaning.py` — this notebook calls it and inspects before/after results for a portfolio-friendly walkthrough).


In [ ]:
import pandas as pd
import sys
sys.path.append('../src')

raw_customers = pd.read_csv('../data/raw/customers.csv')
print(f"Raw customers: {len(raw_customers):,} rows")
raw_customers.head()


## Data quality issues in the raw data

In [ ]:
print("Missing values per column:")
print(raw_customers.isna().sum())
print()
print("Duplicate customer_ids:", raw_customers['customer_id'].duplicated().sum())
print()
print("Distinct gender labels (inconsistent formatting):", raw_customers['gender'].unique())
print()
print("Age range (note invalid values):", raw_customers['age'].min(), "to", raw_customers['age'].max())


## Run the cleaning pipeline

The full, documented cleaning logic is in `src/data_cleaning.py`. Run it from the terminal:

```bash
python src/data_cleaning.py
```

This cell re-runs it here for demonstration.

In [ ]:
import subprocess
result = subprocess.run(['python3', '../src/data_cleaning.py'], capture_output=True, text=True)
print(result.stdout)


## Before / after comparison

In [ ]:
clean_customers = pd.read_csv('../data/processed/customers.csv')

print(f"Raw rows: {len(raw_customers):,}  |  Cleaned rows: {len(clean_customers):,}")
print(f"Raw missing ages: {raw_customers['age'].isna().sum()}  |  Cleaned missing ages: {clean_customers['age'].isna().sum()}")
print(f"Raw gender labels: {sorted(raw_customers['gender'].astype(str).unique())}")
print(f"Cleaned gender labels: {sorted(clean_customers['gender'].astype(str).unique())}")


In [ ]:
with open('../data/processed/cleaning_log.txt') as f:
    print(f.read())


## Summary

Every cleaning decision above is a **documented business rule**, not a silent transformation:
- Duplicates removed by primary key
- Invalid ages nulled then imputed with segment median (not dropped, to preserve customer records)
- Gender/category labels normalized to a consistent set
- Referential integrity enforced (orphaned foreign keys removed) in the order/item/payment/return tables

See `docs/data_dictionary.md` for the full nullable/imputation policy per column.
